# 03 — Dimensão País (`dim_pais`)

Constrói a dimensão de países do modelo estrela da camada gold.

**Chave de junção:** `pais_origem_rfb` — nome exato como aparece na silver (uppercase, sem alterações).
A silver não faz join com a dimensão; o `silver_para_gold.py` valida que todo `pais_origem` existe aqui.

**Cobertura:** a lista de países vem de **toda a série da silver no S3**, não de um único mês.
A versão anterior usava só `estatisticas_di_2026_08.csv` e por isso deixou de fora países que
aparecem apenas em outros períodos (ex.: África do Sul, Austrália, Belarus, Argélia).

**Atenção — espaços:** a RFB publica alguns nomes com espaço duplo (ex.: `ALBANIA, REPUBLICA  DA`).
A chave precisa ser idêntica ao valor da silver, então o dicionário preserva o espaço duplo.
Esta célula também detecta variações de espaço e sinaliza nomes que não batem.

**Saída:** `data_ref/dim_pais.csv` (versionado no repositório).

| Coluna | Descrição |
|---|---|
| `pais_origem_rfb` | Nome exato em uppercase (chave de junção com a silver) |
| `pais_nome` | Nome em português |
| `iso2` / `iso3` | Código ISO 3166-1 alpha-2 / alpha-3 |
| `continente` | Continente |
| `regiao` | Sub-região |
| `bloco_mercosul`, `bloco_ocde`, `bloco_brics`, `bloco_g20` | Pertencimento a blocos (booleano) |

## 0 — Configuração e pasta de saída

In [13]:
import io
import re
from collections import Counter
from pathlib import Path

import boto3
import pandas as pd
import pyarrow.parquet as pq
import yaml

# Pasta de saída: data_ref/ na raiz do projeto
PASTA_DADOS_REF = Path('../../data_ref')
PASTA_DADOS_REF.mkdir(parents=True, exist_ok=True)

with open('../../config.yaml', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

print(f"Pasta de saída: {PASTA_DADOS_REF.resolve()}")

Pasta de saída: K:\Documentos\Profissional\Redes sociais - Portfolio\GitHub\analise-importacoes-brasil\data_ref


## 1 — Países presentes na silver (série completa)

Lê apenas a coluna `pais_origem` de cada Parquet da silver e conta em quantos meses cada país aparece.

In [14]:
session = boto3.Session(profile_name=cfg['aws']['profile'], region_name=cfg['aws']['regiao'])
s3 = session.client('s3')
bucket = cfg['aws']['bucket']
prefixo_silver = cfg['s3']['silver']

chaves = []
for page in s3.get_paginator('list_objects_v2').paginate(Bucket=bucket, Prefix=prefixo_silver):
    chaves += [o['Key'] for o in page.get('Contents', []) if o['Key'].endswith('.parquet')]
print(f"Arquivos na silver: {len(chaves)}")

meses_por_pais = Counter()
for chave in chaves:
    corpo = s3.get_object(Bucket=bucket, Key=chave)['Body'].read()
    paises = pq.read_table(io.BytesIO(corpo), columns=['pais_origem']).column('pais_origem').to_pylist()
    meses_por_pais.update(set(paises))

paises_silver = sorted(meses_por_pais)
print(f"Países distintos na série: {len(paises_silver)}")

Arquivos na silver: 64
Países distintos na série: 122


## 2 — Dicionário de países

Mapeamento manual `pais_origem_rfb → (nome, iso2, iso3, continente, regiao)`.
O `pycountry` não serve aqui: a busca falha com os nomes em português da RFB
(testado: só 22 de 64 bateram, e alguns com erro, como CATAR → Cabo Verde).

> **Manutenção:** quando a célula 3 apontar países novos, adicione-os aqui e reexecute.

In [15]:
DICT_PAISES = {
    # ── AMÉRICAS ──────────────────────────────────────────────────────────────
    "ARGENTINA":                           ("Argentina",          "AR", "ARG", "América do Sul",   "América do Sul"),
    "BARBADOS":                            ("Barbados",           "BB", "BRB", "América do Norte", "Caribe"),
    "BOLIVIA, ESTADO PLURINACIONAL DA":    ("Bolívia",            "BO", "BOL", "América do Sul",   "América do Sul"),
    "BRASIL":                              ("Brasil",             "BR", "BRA", "América do Sul",   "América do Sul"),
    "CANADA":                              ("Canadá",             "CA", "CAN", "América do Norte", "América do Norte"),
    "CHILE":                               ("Chile",              "CL", "CHL", "América do Sul",   "América do Sul"),
    "COLOMBIA":                            ("Colômbia",           "CO", "COL", "América do Sul",   "América do Sul"),
    "COSTA RICA":                          ("Costa Rica",         "CR", "CRI", "América do Norte", "América Central"),
    "EL SALVADOR":                         ("El Salvador",        "SV", "SLV", "América do Norte", "América Central"),
    "EQUADOR":                             ("Equador",            "EC", "ECU", "América do Sul",   "América do Sul"),
    "ESTADOS UNIDOS":                      ("Estados Unidos",     "US", "USA", "América do Norte", "América do Norte"),
    "HONDURAS":                            ("Honduras",           "HN", "HND", "América do Norte", "América Central"),
    "MEXICO":                              ("México",             "MX", "MEX", "América do Norte", "América Central"),
    "PARAGUAI":                            ("Paraguai",           "PY", "PRY", "América do Sul",   "América do Sul"),
    "PERU":                                ("Peru",               "PE", "PER", "América do Sul",   "América do Sul"),
    "PORTO RICO":                          ("Porto Rico",         "PR", "PRI", "América do Norte", "Caribe"),  # território dos EUA; a RFB o trata como origem própria
    "REPUBLICA DOMINICANA":                ("República Dominicana", "DO", "DOM", "América do Norte", "Caribe"),
    "URUGUAI":                             ("Uruguai",            "UY", "URY", "América do Sul",   "América do Sul"),
    "CAYMAN, ILHAS":                       ("Ilhas Cayman",       "KY", "CYM", "América do Norte", "Caribe"),
    "CUBA":                                ("Cuba",               "CU", "CUB", "América do Norte", "Caribe"),
    "GUATEMALA":                           ("Guatemala",          "GT", "GTM", "América do Norte", "América Central"),
    "HAITI":                               ("Haiti",              "HT", "HTI", "América do Norte", "Caribe"),
    "JAMAICA":                             ("Jamaica",            "JM", "JAM", "América do Norte", "Caribe"),
    "NICARAGUA":                           ("Nicarágua",          "NI", "NIC", "América do Norte", "América Central"),
    "PANAMA":                              ("Panamá",             "PA", "PAN", "América do Norte", "América Central"),
    "TRINIDAD E TOBAGO":                   ("Trinidad e Tobago",  "TT", "TTO", "América do Norte", "Caribe"),
    "VENEZUELA":                           ("Venezuela",          "VE", "VEN", "América do Sul",   "América do Sul"),

    # ── EUROPA ────────────────────────────────────────────────────────────────
    "ALBANIA, REPUBLICA  DA":              ("Albânia",            "AL", "ALB", "Europa",           "Europa Meridional"),  # espaço duplo: exatamente como na RFB
    "ALEMANHA":                            ("Alemanha",           "DE", "DEU", "Europa",           "Europa Ocidental"),
    "AUSTRIA":                             ("Áustria",            "AT", "AUT", "Europa",           "Europa Ocidental"),
    "BELARUS, REPUBLICA DA":               ("Belarus",            "BY", "BLR", "Europa",           "Europa Oriental"),
    "BELGICA":                             ("Bélgica",            "BE", "BEL", "Europa",           "Europa Ocidental"),
    "BOSNIA-HERZEGOVINA (REPUBLICA DA)":   ("Bósnia e Herzegovina", "BA", "BIH", "Europa",          "Europa Meridional"),
    "BULGARIA, REPUBLICA DA":              ("Bulgária",           "BG", "BGR", "Europa",           "Europa Oriental"),
    "CROACIA (REPUBLICA DA)":              ("Croácia",            "HR", "HRV", "Europa",           "Europa Meridional"),
    "DINAMARCA":                           ("Dinamarca",          "DK", "DNK", "Europa",           "Europa Setentrional"),
    "ESLOVACA, REPUBLICA":                 ("Eslováquia",         "SK", "SVK", "Europa",           "Europa Oriental"),
    "ESLOVENIA, REPUBLICA DA":             ("Eslovênia",          "SI", "SVN", "Europa",           "Europa Meridional"),
    "ESPANHA":                             ("Espanha",            "ES", "ESP", "Europa",           "Europa Meridional"),
    "ESTONIA, REPUBLICA DA":               ("Estônia",            "EE", "EST", "Europa",           "Europa Setentrional"),
    "FINLANDIA":                           ("Finlândia",          "FI", "FIN", "Europa",           "Europa Setentrional"),
    "FRANCA":                              ("França",             "FR", "FRA", "Europa",           "Europa Ocidental"),
    "GRECIA":                              ("Grécia",             "GR", "GRC", "Europa",           "Europa Meridional"),
    "HUNGRIA, REPUBLICA DA":               ("Hungria",            "HU", "HUN", "Europa",           "Europa Oriental"),
    "IRLANDA":                             ("Irlanda",            "IE", "IRL", "Europa",           "Europa Setentrional"),
    "ITALIA":                              ("Itália",             "IT", "ITA", "Europa",           "Europa Meridional"),
    "LETONIA, REPUBLICA DA":               ("Letônia",            "LV", "LVA", "Europa",           "Europa Setentrional"),
    "LIECHTENSTEIN":                       ("Liechtenstein",      "LI", "LIE", "Europa",           "Europa Ocidental"),
    "LITUANIA, REPUBLICA DA":              ("Lituânia",           "LT", "LTU", "Europa",           "Europa Setentrional"),
    "LUXEMBURGO":                          ("Luxemburgo",         "LU", "LUX", "Europa",           "Europa Ocidental"),
    "MALTA":                               ("Malta",              "MT", "MLT", "Europa",           "Europa Meridional"),
    "NORUEGA":                             ("Noruega",            "NO", "NOR", "Europa",           "Europa Setentrional"),
    "PAISES BAIXOS (HOLANDA)":             ("Países Baixos",      "NL", "NLD", "Europa",           "Europa Ocidental"),
    "POLONIA, REPUBLICA DA":               ("Polônia",            "PL", "POL", "Europa",           "Europa Oriental"),
    "PORTUGAL":                            ("Portugal",           "PT", "PRT", "Europa",           "Europa Meridional"),
    "REINO UNIDO":                         ("Reino Unido",        "GB", "GBR", "Europa",           "Europa Setentrional"),
    "ROMENIA":                             ("Romênia",            "RO", "ROU", "Europa",           "Europa Oriental"),
    "SERVIA":                              ("Sérvia",             "RS", "SRB", "Europa",           "Europa Meridional"),
    "RUSSIA, FEDERACAO DA":                ("Rússia",             "RU", "RUS", "Europa",           "Europa Oriental"),
    "SUECIA":                              ("Suécia",             "SE", "SWE", "Europa",           "Europa Setentrional"),
    "SUICA":                               ("Suíça",              "CH", "CHE", "Europa",           "Europa Ocidental"),
    "TCHECA, REPUBLICA":                   ("República Tcheca",   "CZ", "CZE", "Europa",           "Europa Oriental"),
    "TURQUIA":                             ("Turquia",            "TR", "TUR", "Europa",           "Europa Ocidental"),  # transcontinental: mantida em Europa
    "UCRANIA":                             ("Ucrânia",            "UA", "UKR", "Europa",           "Europa Oriental"),
    "ISLANDIA":                            ("Islândia",           "IS", "ISL", "Europa",           "Europa Setentrional"),
    "MACEDONIA, ANT.REP.IUGOSLAVA":        ("Macedônia do Norte", "MK", "MKD", "Europa",           "Europa Meridional"),
    "MOLDAVIA, REPUBLICA DA":              ("Moldávia",           "MD", "MDA", "Europa",           "Europa Oriental"),
    "MONACO":                              ("Mônaco",             "MC", "MCO", "Europa",           "Europa Ocidental"),

    # ── ÁSIA ──────────────────────────────────────────────────────────────────
    "ARABIA SAUDITA":                      ("Arábia Saudita",     "SA", "SAU", "Ásia",             "Ásia Ocidental"),
    "AZERBAIJAO, REPUBLICA DO":            ("Azerbaijão",         "AZ", "AZE", "Ásia",             "Ásia Ocidental"),
    "BAHREIN, ILHAS":                      ("Bahrein",            "BH", "BHR", "Ásia",             "Ásia Ocidental"),
    "BANGLADESH":                          ("Bangladesh",         "BD", "BGD", "Ásia",             "Ásia Meridional"),
    "CAMBOJA":                             ("Camboja",            "KH", "KHM", "Ásia",             "Ásia Sudoriental"),
    "CAZAQUISTAO, REPUBLICA DO":           ("Cazaquistão",        "KZ", "KAZ", "Ásia",             "Ásia Central"),
    "CATAR":                               ("Catar",              "QA", "QAT", "Ásia",             "Ásia Ocidental"),
    "CHINA, REPUBLICA POPULAR":            ("China",              "CN", "CHN", "Ásia",             "Ásia Oriental"),
    "CINGAPURA":                           ("Cingapura",          "SG", "SGP", "Ásia",             "Ásia Sudoriental"),
    "COREIA (DO SUL), REPUBLICA DA":       ("Coreia do Sul",      "KR", "KOR", "Ásia",             "Ásia Oriental"),
    "EMIRADOS ARABES UNIDOS":              ("Emirados Árabes Unidos", "AE", "ARE", "Ásia",         "Ásia Ocidental"),
    "FILIPINAS":                           ("Filipinas",          "PH", "PHL", "Ásia",             "Ásia Sudoriental"),
    "FORMOSA (TAIWAN)":                    ("Taiwan",             "TW", "TWN", "Ásia",             "Ásia Oriental"),
    "HONG KONG":                           ("Hong Kong",          "HK", "HKG", "Ásia",             "Ásia Oriental"),
    "INDIA":                               ("Índia",              "IN", "IND", "Ásia",             "Ásia Meridional"),
    "INDONESIA":                           ("Indonésia",          "ID", "IDN", "Ásia",             "Ásia Sudoriental"),
    "ISRAEL":                              ("Israel",             "IL", "ISR", "Ásia",             "Ásia Ocidental"),
    "JAPAO":                               ("Japão",              "JP", "JPN", "Ásia",             "Ásia Oriental"),
    "MALASIA":                             ("Malásia",            "MY", "MYS", "Ásia",             "Ásia Sudoriental"),
    "OMA":                                 ("Omã",                "OM", "OMN", "Ásia",             "Ásia Ocidental"),
    "PAQUISTAO":                           ("Paquistão",          "PK", "PAK", "Ásia",             "Ásia Meridional"),
    "SRI LANKA":                           ("Sri Lanka",          "LK", "LKA", "Ásia",             "Ásia Meridional"),
    "TAILANDIA":                           ("Tailândia",          "TH", "THA", "Ásia",             "Ásia Sudoriental"),
    "TURCOMENISTAO, REPUBLICA DO":         ("Turcomenistão",      "TM", "TKM", "Ásia",             "Ásia Central"),
    "UZBEQUISTAO, REPUBLICA DO":           ("Uzbequistão",        "UZ", "UZB", "Ásia",             "Ásia Central"),
    "VIETNA":                              ("Vietnã",             "VN", "VNM", "Ásia",             "Ásia Sudoriental"),
    "COVEITE":                             ("Kuwait",             "KW", "KWT", "Ásia",             "Ásia Ocidental"),
    "GEORGIA, REPUBLICA DA":               ("Geórgia",            "GE", "GEO", "Ásia",             "Ásia Ocidental"),
    "IRA, REPUBLICA ISLAMICA DO":          ("Irã",                "IR", "IRN", "Ásia",             "Ásia Ocidental"),
    "JORDANIA":                            ("Jordânia",           "JO", "JOR", "Ásia",             "Ásia Ocidental"),
    "LAOS, REP.POP.DEMOCR.DO":             ("Laos",               "LA", "LAO", "Ásia",             "Ásia Sudoriental"),
    "LIBANO":                              ("Líbano",             "LB", "LBN", "Ásia",             "Ásia Ocidental"),
    "MIANMAR (BIRMANIA)":                  ("Mianmar",            "MM", "MMR", "Ásia",             "Ásia Sudoriental"),
    "MONGOLIA":                            ("Mongólia",           "MN", "MNG", "Ásia",             "Ásia Oriental"),

    # ── ÁFRICA ────────────────────────────────────────────────────────────────
    "AFRICA DO SUL":                       ("África do Sul",      "ZA", "ZAF", "África",           "África Meridional"),
    "ARGELIA":                             ("Argélia",            "DZ", "DZA", "África",           "África do Norte"),
    "BURKINA FASO":                        ("Burkina Faso",       "BF", "BFA", "África",           "África Ocidental"),
    "CAMAROES":                            ("Camarões",           "CM", "CMR", "África",           "África Central"),
    "COSTA DO MARFIM":                     ("Costa do Marfim",    "CI", "CIV", "África",           "África Ocidental"),
    "EGITO":                               ("Egito",              "EG", "EGY", "África",           "África do Norte"),
    "MADAGASCAR":                          ("Madagascar",         "MG", "MDG", "África",           "África Oriental"),
    "MARROCOS":                            ("Marrocos",           "MA", "MAR", "África",           "África do Norte"),
    "NIGERIA":                             ("Nigéria",            "NG", "NGA", "África",           "África Ocidental"),
    "QUENIA":                              ("Quênia",             "KE", "KEN", "África",           "África Oriental"),
    "TUNISIA":                             ("Tunísia",            "TN", "TUN", "África",           "África do Norte"),
    "COMORES, ILHAS":                      ("Comores",            "KM", "COM", "África",           "África Oriental"),
    "GANA":                                ("Gana",               "GH", "GHA", "África",           "África Ocidental"),
    "LIBIA":                               ("Líbia",              "LY", "LBY", "África",           "África do Norte"),
    "MARSHALL,ILHAS":                      ("Ilhas Marshall",     "MH", "MHL", "Oceania",          "Micronésia"),
    "MAURICIO":                            ("Maurícia",           "MU", "MUS", "África",           "África Oriental"),
    "SUDAO":                               ("Sudão",              "SD", "SDN", "África",           "África do Norte"),
    "TOGO":                                ("Togo",               "TG", "TGO", "África",           "África Ocidental"),

    # ── OCEANIA ───────────────────────────────────────────────────────────────
    "AUSTRALIA":                           ("Austrália",          "AU", "AUS", "Oceania",          "Austrália e Nova Zelândia"),
    "NOVA ZELANDIA":                       ("Nova Zelândia",      "NZ", "NZL", "Oceania",          "Austrália e Nova Zelândia"),
}

print(f"Países no dicionário: {len(DICT_PAISES)}")

Países no dicionário: 122


## 3 — Conferência: cobertura e variações de nome

- **Sem correspondência:** países da silver que não estão no dicionário. Devem ser zero antes de salvar.
- **Variação de espaço:** nome da silver que bate com uma chave do dicionário só depois de colapsar espaços duplos.
- **Não usados:** chaves do dicionário que não aparecem na silver. Não são erro, mas merecem revisão.

In [16]:
def normalizar_espacos(texto: str) -> str:
    return re.sub(r'\s+', ' ', texto).strip()

chaves_dict = set(DICT_PAISES)
sem_correspondencia = [p for p in paises_silver if p not in chaves_dict]

variacoes = []
for p in sem_correspondencia:
    alvo = [k for k in chaves_dict if normalizar_espacos(k) == normalizar_espacos(p)]
    if alvo:
        variacoes.append((p, alvo[0]))

nao_usados = sorted(chaves_dict - set(paises_silver))

print(f"Países na silver          : {len(paises_silver)}")
print(f"Sem correspondência       : {len(sem_correspondencia)}")
for p in sem_correspondencia:
    print(f"   '{p}'  (meses: {meses_por_pais[p]})")
print(f"\nVariações de espaço       : {len(variacoes)}")
for p, alvo in variacoes:
    print(f"   '{p}'  ≈  '{alvo}'")
print(f"\nNo dicionário, fora da série: {len(nao_usados)}")
for k in nao_usados:
    print(f"   '{k}'")

if sem_correspondencia:
    raise ValueError(f"{len(sem_correspondencia)} país(es) sem correspondência. "
                     "Adicione-os à célula 2 e reexecute.")
print("\n✅ Todos os países da série têm correspondência no dicionário.")

Países na silver          : 122
Sem correspondência       : 0

Variações de espaço       : 0

No dicionário, fora da série: 0

✅ Todos os países da série têm correspondência no dicionário.


## 4 — Blocos econômicos

Definidos por código ISO2, para ficar consistente com o dicionário.

In [17]:
# Mercosul: membros plenos + associados
MERCOSUL = {"AR", "BR", "UY", "PY", "BO", "CL", "CO", "EC", "PE"}

# OCDE: membros (inclui Costa Rica, membro desde 2021)
OCDE = {
    "AU", "AT", "BE", "CA", "CL", "CO", "CR", "CZ", "DK", "EE",
    "FI", "FR", "DE", "GR", "HU", "IS", "IE", "IL", "IT",
    "JP", "KR", "LV", "LT", "LU", "MX", "NL", "NZ", "NO",
    "PL", "PT", "SK", "SI", "ES", "SE", "CH", "TR", "GB", "US",
}

# BRICS: originais + membros de 2024 (EG, SA, AE, ET, IR; ET e IR não aparecem na série)
BRICS = {"BR", "RU", "IN", "CN", "ZA", "EG", "SA", "AE"}

# G20: países membros (UE: principais membros europeus que aparecem na série)
G20 = {
    "AR", "AU", "BR", "CA", "CN", "FR", "DE", "IN", "ID",
    "IT", "JP", "KR", "MX", "RU", "SA", "ZA", "TR", "GB", "US",
    "ES", "NL", "PL",
}

print(f"Mercosul: {len(MERCOSUL)} | OCDE: {len(OCDE)} | BRICS: {len(BRICS)} | G20: {len(G20)}")

Mercosul: 9 | OCDE: 38 | BRICS: 8 | G20: 22


## 5 — Construção do DataFrame

In [18]:
registros = []
for pais_rfb, (nome, iso2, iso3, continente, regiao) in DICT_PAISES.items():
    registros.append({
        "pais_origem_rfb": pais_rfb,
        "pais_nome":       nome,
        "iso2":            iso2,
        "iso3":            iso3,
        "continente":      continente,
        "regiao":          regiao,
        "bloco_mercosul":  iso2 in MERCOSUL,
        "bloco_ocde":      iso2 in OCDE,
        "bloco_brics":     iso2 in BRICS,
        "bloco_g20":       iso2 in G20,
    })

dim_pais = (pd.DataFrame(registros)
              .sort_values(["continente", "pais_nome"])
              .reset_index(drop=True))

print(f"Linhas: {len(dim_pais)} | Colunas: {len(dim_pais.columns)}")
dim_pais.head(10)

Linhas: 122 | Colunas: 10


,pais_origem_rfb,pais_nome,iso2,iso3,continente,regiao,bloco_mercosul,bloco_ocde,bloco_brics,bloco_g20
0,BARBADOS,Barbados,BB,BRB,América do Norte,Caribe,False,False,False,False
1,CANADA,Canadá,CA,CAN,América do Norte,América do Norte,False,True,False,True
2,COSTA RICA,Costa Rica,CR,CRI,América do Norte,América Central,False,True,False,False
3,CUBA,Cuba,CU,CUB,América do Norte,Caribe,False,False,False,False
4,EL SALVADOR,El Salvador,SV,SLV,América do Norte,América Central,False,False,False,False
5,ESTADOS UNIDOS,Estados Unidos,US,USA,América do Norte,América do Norte,False,True,False,True
6,GUATEMALA,Guatemala,GT,GTM,América do Norte,América Central,False,False,False,False
7,HAITI,Haiti,HT,HTI,América do Norte,Caribe,False,False,False,False
8,HONDURAS,Honduras,HN,HND,América do Norte,América Central,False,False,False,False
9,"CAYMAN, ILHAS",Ilhas Cayman,KY,CYM,América do Norte,Caribe,False,False,False,False


## 6 — Checagens de qualidade

In [19]:
erros, avisos = [], []

# Schema
for col in ["pais_origem_rfb", "pais_nome", "iso2", "iso3", "continente", "regiao"]:
    n = dim_pais[col].isna().sum() + (dim_pais[col] == "").sum()
    if n:
        erros.append(f"Nulos ou vazios em '{col}': {n}")

dup = dim_pais["pais_origem_rfb"].duplicated().sum()
if dup:
    erros.append(f"pais_origem_rfb com {dup} duplicata(s)")

if (dim_pais["iso2"].str.len() != 2).any():
    erros.append("iso2 com tamanho diferente de 2")
if (dim_pais["iso3"].str.len() != 3).any():
    erros.append("iso3 com tamanho diferente de 3")

if (dim_pais["pais_origem_rfb"] != dim_pais["pais_origem_rfb"].str.upper()).any():
    erros.append("pais_origem_rfb fora do uppercase")

# Negócio
CONTINENTES = {"África", "América do Norte", "América do Sul", "Ásia", "Europa", "Oceania"}
invalidos = set(dim_pais["continente"]) - CONTINENTES
if invalidos:
    erros.append(f"Continentes inválidos: {invalidos}")

iso2_dup = dim_pais[dim_pais.duplicated("iso2", keep=False)]
if len(iso2_dup):
    avisos.append(f"iso2 repetido (mesmo código, nomes diferentes): {iso2_dup['pais_origem_rfb'].tolist()}")

# Cobertura da série
faltando = set(paises_silver) - set(dim_pais["pais_origem_rfb"])
if faltando:
    erros.append(f"Países da série sem linha na dimensão: {sorted(faltando)}")

print("=" * 60)
print("RELATÓRIO DE QUALIDADE — dim_pais")
print("=" * 60)
if not erros and not avisos:
    print("✅ Sem erros ou avisos.")
for e in erros:
    print("❌", e)
for a in avisos:
    print("⚠️ ", a)

print(f"\nTotal de países: {len(dim_pais)}")
print("\nPor continente:")
print(dim_pais["continente"].value_counts().to_string())
print("\nMembros de blocos:")
for col in ["bloco_mercosul", "bloco_ocde", "bloco_brics", "bloco_g20"]:
    print(f"  {col:<16} {int(dim_pais[col].sum())}")

if erros:
    raise ValueError(f"dim_pais com {len(erros)} erro(s). Corrija antes de salvar.")

RELATÓRIO DE QUALIDADE — dim_pais
✅ Sem erros ou avisos.

Total de países: 122

Por continente:
continente
Europa              41
Ásia                34
América do Norte    17
África              17
América do Sul      10
Oceania              3

Membros de blocos:
  bloco_mercosul   9
  bloco_ocde       38
  bloco_brics      8
  bloco_g20        22


## 7 — Salvar `dim_pais.csv`

In [20]:
caminho_saida = PASTA_DADOS_REF / "dim_pais.csv"
dim_pais.to_csv(caminho_saida, index=False, encoding="utf-8-sig")
print(f"✅ Salvo: {caminho_saida.resolve()} ({len(dim_pais)} linhas)")

✅ Salvo: K:\Documentos\Profissional\Redes sociais - Portfolio\GitHub\analise-importacoes-brasil\data_ref\dim_pais.csv (122 linhas)


## 8 — Como a gold usa esta dimensão

O `silver_para_gold.py` valida que todo `pais_origem` da silver existe em `pais_origem_rfb`
e grava a dimensão em `gold/estatisticas_di/dim_pais/`. A fato guarda apenas a chave; o relacionamento
com `dim_pais` é feito no modelo (Power BI) ou no join das análises.